# Fig. — min-SINR CDF + SCNR CDF at $\gamma^\star$

Interactive front-end for the two-panel CDF figure. The plotting logic lives in
`build_fig_cdf.py` (the headless builder); this notebook just imports and reuses
`load_cdf`, `build_figure`, `collect_summary`, `_present_algorithms`, and
`_select_scnr_metric`, so the notebook and the committed PDF can never drift.

- **(a)** min-SINR CDF at $\gamma^\star$ — mass left of the $\gamma^\star$ line is the
  per-user outage; the legend annotates each algorithm's infeasibility rate.
- **(b)** SCNR CDF at the same operating point — the sensing performance you get
  once the communication floor is enforced at $\gamma^\star$.

**Compatible experiments:** `sinr_cdf` (panel a), `scnr_cdf` (panel b). If only one
campaign has finished, the builder reuses it for both panels.

In [ ]:
# Make the builder + cordis importable, then apply the paper rcParams.
import sys, logging
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

FIG_DIR = Path.cwd()
if str(FIG_DIR) not in sys.path:
    sys.path.insert(0, str(FIG_DIR))

import build_fig_cdf as B   # the builder module — single source of truth
from cordis.plotting import apply_paper_style, save_figure

# Set use_tex=False below if pdflatex isn't on PATH (e.g. a compute node).
apply_paper_style()
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

## 1. Load the runs

Leave `SINR_DIR` / `SCNR_DIR` as `None` to auto-pick the newest run of each
experiment (an aggregated array run, `array_<jobid>_aggregated/`, is preferred;
the unreliable `latest` symlink is never used). Pin a path to re-render an older
campaign.

In [ ]:
SINR_DIR = None   # e.g. 'results/exp_sinr_cdf/array_12345_aggregated'
SCNR_DIR = None   # e.g. 'results/exp_scnr_cdf/array_12346_aggregated'

sinr_result, sinr_dir = B.load_cdf(SINR_DIR, experiment='sinr_cdf')
try:
    scnr_result, scnr_dir = B.load_cdf(SCNR_DIR, experiment='scnr_cdf')
except FileNotFoundError:
    print('No scnr_cdf run; reusing the sinr_cdf run for panel (b).')
    scnr_result, scnr_dir = sinr_result, sinr_dir
print('SINR run:', sinr_dir)
print('SCNR run:', scnr_dir)

## 2. Resolve the operating point and the algorithm/metric selection

`GAMMA_DB = None` auto-detects $\gamma^\star$ from the run metadata (falling back
to 0 dB only if absent). Override it explicitly for a paper-grade figure if you
want the caption number pinned.

In [ ]:
GAMMA_DB    = None                         # None -> auto-detect from metadata
ONLY        = list(B.PREFERRED)            # ['Centralized','CORDIS-ADMM','CORDIS-Split']
SCNR_METRIC = None                         # None -> first available of B.SCNR_METRIC_PREFERENCE

gamma_db, detected = (GAMMA_DB, True) if GAMMA_DB is not None \
    else B._detect_gamma_db(sinr_result)
only        = [n for n in ONLY if n in (sinr_result.sim_result.names)]
if not only:
    only = B._present_algorithms(sinr_result, B.PREFERRED)
scnr_metric = SCNR_METRIC or B._select_scnr_metric(scnr_result, only=only)
print(f'gamma* = {gamma_db:g} dB  (auto-detected={detected})')
print(f'algorithms = {only}')
print(f'scnr metric = {scnr_metric}')

## 3. Numeric summary (caption sanity check)

In [ ]:
summary = B.collect_summary(sinr_result, scnr_result, gamma_db, only, scnr_metric)
B._print_summary(summary, gamma_db, scnr_metric)

## 4. Build the figure (from the module)

This calls the builder's `build_figure` verbatim — identical to the committed PDF.
Edit `build_fig_cdf.py` (not a copy here) so the headless build stays in sync.

In [ ]:
USE_TEX = True   # False on a node without pdflatex
fig, used_only, used_scnr_metric = B.build_figure(
    sinr_result, scnr_result,
    gamma_db=gamma_db, only=only,
    scnr_metric=scnr_metric, use_tex=USE_TEX,
)
plt.show()

## 5. Editable copy of `build_figure`

This is a verbatim copy of the module's `build_figure`, renamed `build_figure_editable`
so it shadows the import only when you call it. Edit the plotting here (axes, legend,
colors, which metrics) and re-run to explore. When you're happy, port the change into
`build_fig_cs_tradeoff.py` so the committed figure matches.

In [ ]:
def build_figure_editable(sinr_result,
                 scnr_result,
                 *,
                 gamma_db: float,
                 only: B.Optional[B.Sequence[str]] = None,
                 scnr_metric: B.Optional[str] = None,
                 use_tex: bool = True):
    """Assemble the two-panel CDF figure and return the matplotlib Figure.

    Reuses ``cordis.plotting.plot_cdf`` for both panels so styling, the γ
    marker, and the infeasibility annotation match the rest of the toolkit.
    """
    import matplotlib
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False
    import matplotlib.pyplot as plt
    from cordis.plotting import apply_paper_style, figsize, plot_cdf

    apply_paper_style()
    if use_tex is False:
        # apply_paper_style may flip usetex back on; force it off for nodes
        # without a TeX install.
        matplotlib.rcParams["text.usetex"] = False

    only = list(only) if only else B._present_algorithms(sinr_result, B.PREFERRED)
    if scnr_metric is None:
        scnr_metric = B._select_scnr_metric(scnr_result, only=only)

    have_scnr = scnr_metric is not None
    ncols = 2 if have_scnr else 1
    fig, axes = plt.subplots(
        1, ncols,
        figsize=figsize(width="double" if have_scnr else "single",
                        aspect=(7.16 / 2.8) if have_scnr else (3.5 / 2.6)),
    )
    axes = np.atleast_1d(axes)

    # ── Panel (a): min-SINR CDF at γ* ───────────────────────────────
    ax_a = axes[0]
    plot_cdf(
        sinr_result.sim_result,
        metric=B.SINR_METRIC,
        ax=ax_a,
        xlabel=r"min-SINR [dB]",
        only=only,
        gamma_db=gamma_db,            # vertical γ* line + infeasibility annotation
        annotate_infeasibility=True,
        legend_loc="lower right",     # any CDF is empty in the lower-right corner
    )
    ax_a.set_ylim(0.0, 1.0)
    ax_a.set_title(r"(a) min-SINR CDF at $\gamma^\star$")

    # ── Panel (b): SCNR CDF at γ* ───────────────────────────────────
    if have_scnr:
        ax_b = axes[1]
        plot_cdf(
            scnr_result.sim_result,
            metric=scnr_metric,
            ax=ax_b,
            xlabel=B._SCNR_XLABEL.get(scnr_metric, scnr_metric),
            only=only,
            legend_loc="lower right",
        )
        ax_b.set_ylim(0.0, 1.0)
        ax_b.set_title(r"(b) SCNR CDF at $\gamma^\star$")

    fig.tight_layout()
    return fig, only, scnr_metric

In [ ]:
USE_TEX = True   # False on a node without pdflatex
fig, used_only, used_scnr_metric = build_figure_editable(
    sinr_result, scnr_result,
    gamma_db=gamma_db, only=only,
    scnr_metric=scnr_metric, use_tex=USE_TEX,
)
plt.show()

## 5. Save to `paper/figures/`

Writes the committed PDF the LaTeX `\includegraphics` reads, with provenance
metadata embedded.

In [ ]:
out_stem = FIG_DIR.parents[1] / 'figures' / 'fig_cdf'
paths = save_figure(
    fig, out_stem, formats=('pdf',),
    metadata={
        'Figure': 'fig_cdf',
        'GammaStarDB': f'{gamma_db:g}',
        'Algorithms': ', '.join(used_only),
        'ScnrMetric': str(used_scnr_metric),
        'SinrRun': sinr_dir.name,
        'ScnrRun': scnr_dir.name,
    },
)
for p in paths:
    print('wrote', p)